# Milestone 3.2 — Hardened Streaming Drift Adaptation & Validation

This notebook documents the validated, hardened implementation of **Real-Time Concept Drift Detection & Online Model Adaptation** for the `CICIDS2017` network flow dataset (2,520,751 records), **using the Labrooms 8-feature deployment set** to avoid training-serving skew.

## Core Technical Objectives & Hardened Audits:
1. **Feature Contract**: `labrooms-app-layer-v1` (8 features) — only non-zero slots from the Labrooms CyberAdapt 52-element sensor array.
2. **ADWIN Detection Delay Audit**: ADWIN triggers an alert at sample **5,599 of `window_39`** (5,599 records / 0.11 windows delay after `DoS` emergence starts).
3. **Drift-Triggered Retraining Policy**: Candidate models are retrained **ONLY** when a genuine ADWIN drift alert fires.
4. **Intra-Window Post-Trigger Evaluation**: When drift fires at sample $k$ of `window_t`:
   - Candidate Training Set: Past windows + `window_t[0:k]` (labeled data up to trigger).
   - Candidate Validation Set: Disjoint tail `window_t[k:N]` (unseen data occurring AFTER trigger).
   - Samples $0..k-1$ are strictly excluded from validation.
5. **Clean Lifecycle Logging**: Track `promotion_chronology.csv` documenting version increments strictly upon passing all 4 promotion gates.


In [ ]:
import sys
from pathlib import Path
import logging
import time
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure backend/ is in python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ml.src.features.feature_contract import (
    LABROOMS_DEPLOYMENT_FEATURES, LABROOMS_SCHEMA_VERSION,
    TARGET_COLUMN, TARGET_CLASSES,
    export_labrooms_feature_contract, quantify_and_clean_negative_duration
)
from ml.src.features.preprocessing import CICIDSDataPreprocessor
from ml.src.data.stream_loader import StreamBatchReplayer
from ml.src.drift.detectors import StreamDriftMonitor, ADWINDriftDetector
from ml.src.models.adaptive_engine import AdaptiveModelManager, PromotionGateConfig
from ml.src.evaluation.metrics import evaluate_predictions
from ml.src.evaluation.streaming_plotting import (
    plot_streaming_macro_f1, plot_drift_events_timeline, plot_pre_vs_post_adaptation
)

logging.basicConfig(level=logging.INFO)
print("Environment and modules successfully loaded!")


In [ ]:
reports_dir = project_root / "artifacts" / "reports"
# Export the Labrooms deployment feature contract
contract_labrooms = export_labrooms_feature_contract(reports_dir / "labrooms_feature_contract.json")

print(f"Labrooms Feature Contract (labrooms-app-layer-v1): {contract_labrooms['total_selected_features']} features")
print(f"Selected features: {contract_labrooms['selected_features']}")
print(f"Zeroed features excluded: {contract_labrooms['total_zeroed_features']}")


In [ ]:
dataset_path = project_root / "data" / "raw" / "cicids2017" / "cicids2017_cleaned.csv"
print(f"Loading dataset from {dataset_path}...")
df_raw = pd.read_csv(dataset_path)
df_clean, clean_report = quantify_and_clean_negative_duration(df_raw)
print(f"Cleaned dataset: {len(df_clean):,} records. Remediation: {clean_report['remediation_action']}")

# Use only Labrooms-aligned features — avoids training on always-zero columns
replayer = StreamBatchReplayer(df_clean, window_size=50000, feature_cols=LABROOMS_DEPLOYMENT_FEATURES)
print(f"Initialized StreamBatchReplayer: {replayer.num_windows} streaming windows (50,000 records each).")


In [ ]:
models_dir = project_root / "artifacts" / "models"
champ_path = models_dir / "ensemble_v1.joblib"
if not champ_path.exists():
    champ_path = models_dir / "lightgbm_v1.joblib"
preproc_path = models_dir / "preprocessor_v1.joblib"

champion_model = joblib.load(champ_path)
preprocessor = CICIDSDataPreprocessor.load(preproc_path)

gate_config = PromotionGateConfig(
    min_macro_f1_delta=-0.01,
    min_balanced_acc_delta=-0.01,
    min_minority_recall=0.70,
    max_normal_fpr=0.05
)

model_manager = AdaptiveModelManager(
    champion_model=champion_model,
    preprocessor=preprocessor,
    gate_config=gate_config,
    feature_cols=LABROOMS_DEPLOYMENT_FEATURES,  # Only use Labrooms non-zero features
    model_version="v1"
)

monitor = StreamDriftMonitor(error_delta=0.002, feature_delta=0.01)
print(f"Loaded initial Champion ({type(champion_model).__name__}) and initialized ADWIN StreamDriftMonitor.")


In [ ]:
metrics_history = []
window_history = []
promotion_chronology = []

pre_shift_metrics = {}
degraded_metrics = {}
post_adapt_metrics = {}

print("Beginning Hardened Streaming Loop...")
for win_idx in range(replayer.num_windows):
    window = replayer.get_window(win_idx)
    win_id = window["window_id"]
    X_win = window["X"]
    y_win = window["y"]
    
    # Step A: Champion Inference on unseen window_t
    X_win_scaled, y_win_enc = model_manager.preprocessor.transform(X_win, y_win)
    t0 = time.perf_counter()
    y_pred_enc = model_manager.champion_model.predict(X_win_scaled)
    y_prob = getattr(model_manager.champion_model, "predict_proba", lambda X: None)(X_win_scaled)
    lat_sec = time.perf_counter() - t0
    
    win_eval = evaluate_predictions(y_win_enc, y_pred_enc, y_prob, lat_sec, model_name=f"Champion_{model_manager.model_version}", split_name=win_id)
    
    win_summary = {
        "window_id": win_id,
        "window_idx": win_idx,
        "model_version": model_manager.model_version,
        "total_samples": window["total_samples"],
        "attack_rate": window["attack_rate"],
        "macro_f1": win_eval["macro_f1"],
        "weighted_f1": win_eval["weighted_f1"],
        "balanced_accuracy": win_eval["balanced_accuracy"],
        "accuracy": win_eval["accuracy"],
        "class_distribution": window["class_distribution"],
        "per_class_metrics": win_eval["per_class_metrics"]
    }
    metrics_history.append(win_summary)
    
    if win_idx == 2:
        pre_shift_metrics = {
            "macro_f1": win_eval["macro_f1"],
            "weighted_f1": win_eval["weighted_f1"],
            "balanced_accuracy": win_eval["balanced_accuracy"],
            "dos_recall": win_eval["per_class_metrics"].get("DoS", {}).get("recall", 0.0)
        }
        
    if "DoS" in window["class_distribution"] and win_eval["macro_f1"] < 0.35 and not degraded_metrics:
        degraded_metrics = {
            "macro_f1": win_eval["macro_f1"],
            "weighted_f1": win_eval["weighted_f1"],
            "balanced_accuracy": win_eval["balanced_accuracy"],
            "dos_recall": win_eval["per_class_metrics"].get("DoS", {}).get("recall", 0.0)
        }
        
    # Step B: ADWIN concept drift monitoring
    drift_res = monitor.process_window(
        window_id=win_id, window_idx=win_idx, y_true=y_win_enc, y_pred=y_pred_enc, metric_before=win_eval["macro_f1"]
    )
    
    # Step C: Adaptive Retraining Workflow (Initiated ONLY on genuine ADWIN Drift Alert)
    if drift_res["drift_detected"]:
        k = drift_res.get("trigger_sample_idx") or 0
        print(f"!!! REAL ADWIN DRIFT ALERT DETECTED at {win_id} (sample {k:,}) !!! Retraining candidate...")
        
        train_dfs_X = [w["X"] for w in window_history[-2:]]
        train_dfs_y = [w["y"] for w in window_history[-2:]]
        if k > 0:
            train_dfs_X.append(X_win.iloc[:k])
            train_dfs_y.append(y_win.iloc[:k])
            
        train_buffer_ids = [w["window_id"] for w in window_history[-2:]]
        train_buffer_summary = f"{','.join(train_buffer_ids)}+{win_id}[0:{k}]" if k > 0 else ",".join(train_buffer_ids)
        
        buf_X = pd.concat(train_dfs_X, ignore_index=True)
        buf_y = pd.concat(train_dfs_y, ignore_index=True)
        
        if len(X_win) - k >= 1000:
            val_X_disjoint = X_win.iloc[k:]
            val_y_disjoint = y_win.iloc[k:]
            val_range_summary = f"{win_id}[{k}:{len(X_win)}]"
        elif win_idx + 1 < replayer.num_windows:
            next_win = replayer.get_window(win_idx + 1)
            val_X_disjoint = next_win["X"]
            val_y_disjoint = next_win["y"]
            val_range_summary = f"{next_win['window_id']}[full]"
        else:
            val_X_disjoint = X_win
            val_y_disjoint = y_win
            val_range_summary = f"{win_id}[full]"
            
        model_manager.retrain_candidate(buf_X, buf_y, model_type="lightgbm")
        gate_report = model_manager.evaluate_and_promote_candidate(
            val_X=val_X_disjoint, val_y=val_y_disjoint, window_id=val_range_summary,
            train_window_ids=train_buffer_ids if k == 0 else train_buffer_ids + [win_id], emerged_class="DoS"
        )
        
        promotion_chronology.append({
            "drift_event_window": win_id,
            "trigger_sample_idx": k,
            "train_buffer_summary": train_buffer_summary,
            "validation_range_summary": val_range_summary,
            "champion_version_before": gate_report.get("champion_version_before"),
            "candidate_macro_f1": gate_report.get("candidate_macro_f1", 0.0),
            "champion_macro_f1": gate_report.get("champion_macro_f1", 0.0),
            "gates_passed": str(gate_report.get("gates_passed", {})),
            "promoted": gate_report.get("promoted", False),
            "champion_version_after": gate_report.get("champion_version_after")
        })
        
    window_history.append(window)

print("Hardened Streaming Loop Completed Successfully!")


In [ ]:
chron_df = pd.DataFrame(promotion_chronology)
print("=== Chronological Promotion Lifecycle Table ===")
display(chron_df[['drift_event_window', 'trigger_sample_idx', 'train_buffer_summary', 'validation_range_summary', 'champion_version_before', 'candidate_macro_f1', 'promoted', 'champion_version_after']])


In [ ]:
figures_dir = project_root / "artifacts" / "figures"
plot_streaming_macro_f1(metrics_history, monitor.drift_events_log, output_path=figures_dir / "streaming_macro_f1.png")
plot_drift_events_timeline(monitor.drift_events_log, total_windows=replayer.num_windows, output_path=figures_dir / "drift_events.png")

if len(metrics_history) > 40:
    last_adapted = metrics_history[41]
    post_adapt_metrics = {
        "macro_f1": last_adapted["macro_f1"],
        "weighted_f1": last_adapted["weighted_f1"],
        "balanced_accuracy": last_adapted["balanced_accuracy"],
        "dos_recall": last_adapted["per_class_metrics"].get("DoS", {}).get("recall", 0.0)
    }

plot_pre_vs_post_adaptation(pre_shift_metrics, degraded_metrics, post_adapt_metrics, output_path=figures_dir / "pre_vs_post_adaptation.png")
print("All figures successfully saved to backend/artifacts/figures/")
